# Pooling Ablation — Text-Only RoBERTa AES (CLS vs Mean vs Max)

**Hyperparameters tetap:** `lr=2e-5`, `dropout=0.2`, `hidden_dim=256`, `epochs=15`.

In [1]:
import os
import re
import random
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import GroupShuffleSplit

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

def set_seed(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    try:
        if hasattr(torch, 'xpu') and torch.xpu.is_available():
            torch.xpu.manual_seed(seed)
            torch.xpu.manual_seed_all(seed)
    except ImportError:
        print("Warning: intel_extension_for_pytorch (ipex) tidak ditemukan.")
    torch.use_deterministic_algorithms(True, warn_only=True)

set_seed(42)

def get_device():
    if torch.cuda.is_available(): return "cuda"
    try:
        if torch.xpu.is_available(): return "xpu"
    except: pass
    if torch.backends.mps.is_available(): return "mps"
    return "cpu"

DEVICE = get_device()
print(f"Using Device: {DEVICE}")

c:\Users\gabyg\miniconda3\envs\skripsi\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using Device: xpu


## Konfigurasi

In [ ]:
CSV_FILE   = "../../data.csv"
BATCH_SIZE = 8

# Hyperparameters TETAP (sama dgn rq4/train_text_only)
LEARNING_RATE = 2e-5
DROPOUT_RATE  = 0.2
HIDDEN_DIM    = 256
EPOCHS        = 15

# Pooling strategies yang dibandingkan
POOLING_STRATEGIES = ["cls", "mean", "max"]

TEXT_TRAITS = [
    'organizational_structure(ground_truth)', 'coherence(ground_truth)',
    'essay_length(ground_truth)', 'grammatical_accuracy(ground_truth)',
    'grammatical_diversity(ground_truth)', 'lexical_accuracy(ground_truth)',
    'lexical_diversity(ground_truth)', 'punctuation_accuracy(ground_truth)',
]

RESULTS_CSV = 'pooling_ablation_text_only_results.csv'

print(f"LR={LEARNING_RATE} | Drop={DROPOUT_RATE} | HiddenDim={HIDDEN_DIM} | Epochs={EPOCHS}")
print(f"Pooling : {POOLING_STRATEGIES}")
print(f"Traits ({len(TEXT_TRAITS)}): {[t.replace('(ground_truth)','') for t in TEXT_TRAITS]}")

LR=2e-05 | Drop=0.2 | HiddenDim=256 | Epochs=15
Pooling : ['max']
Traits (8): ['organizational_structure', 'coherence', 'essay_length', 'grammatical_accuracy', 'grammatical_diversity', 'lexical_accuracy', 'lexical_diversity', 'punctuation_accuracy']


## Dataset

In [3]:
class TextOnlyDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.df = df
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        essay_text  = str(row.get('Essay', ""))
        prompt_text = str(row.get('Question', ""))

        enc = self.tokenizer(
            prompt_text, essay_text,
            max_length=512, padding="max_length", truncation=True, return_tensors="pt"
        )
        labels = [row[c] for c in TEXT_TRAITS]

        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels_txt':     torch.tensor(labels, dtype=torch.float32),
        }

## Model: RobertaSemanticAES dgn pooling strategy yang bisa diganti

- **cls**  : ambil token `[CLS]` (`last_hidden_state[:, 0, :]`)
- **mean** : masked mean pooling — sama dgn baseline rq4/train_text_only
- **max**  : masked max pooling — token padding di-set `-inf` agar tidak ikut di-max

In [4]:
class RobertaSemanticAES(nn.Module):
    def __init__(self, dropout_rate=DROPOUT_RATE, hidden_dim=HIDDEN_DIM, pooling="mean"):
        super().__init__()
        assert pooling in ("cls", "mean", "max"), f"Pooling tidak dikenal: {pooling}"
        self.pooling = pooling

        self.roberta = AutoModel.from_pretrained("roberta-base")

        for p in self.roberta.parameters():
            p.requires_grad = False
        for layer in self.roberta.encoder.layer[-6:]:
            for p in layer.parameters():
                p.requires_grad = True

        self.text_dim = 768
        self.dropout  = nn.Dropout(dropout_rate)

        self.txt_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.text_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout_rate),
                nn.Linear(hidden_dim, 1),
            ) for _ in range(len(TEXT_TRAITS))
        ])

    def _pool(self, last_hidden_state, attention_mask):
        if self.pooling == "cls":
            return last_hidden_state[:, 0, :]

        mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()

        if self.pooling == "mean":
            sum_emb  = torch.sum(last_hidden_state * mask, dim=1)
            sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
            return sum_emb / sum_mask

        # max
        masked = last_hidden_state.masked_fill(mask == 0, float('-inf'))
        return masked.max(dim=1).values

    def forward(self, input_ids, attention_mask):
        txt_out    = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        bert_embed = self._pool(txt_out.last_hidden_state, attention_mask)
        bert_embed = self.dropout(bert_embed)
        txt_preds  = [h(bert_embed) for h in self.txt_heads]
        return torch.cat(txt_preds, dim=1)

## Helpers: Evaluate QWK & GPU Cleanup

In [5]:
def evaluate_qwk(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for b in dataloader:
            ids  = b['input_ids'].to(DEVICE)
            mask = b['attention_mask'].to(DEVICE)
            lbl  = b['labels_txt'].cpu().numpy()

            preds = model(ids, mask).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(lbl)

    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    qwk_scores = []
    for i in range(len(TEXT_TRAITS)):
        pred_classes = np.round(all_preds[:, i]  * 2).astype(int)
        gt_classes   = np.round(all_labels[:, i] * 2).astype(int)
        try:
            qwk = cohen_kappa_score(gt_classes, pred_classes, weights='quadratic')
            if np.isnan(qwk): qwk = 0.0
        except:
            qwk = 0.0
        qwk_scores.append(qwk)
    return qwk_scores


def _clear_gpu():
    gc.collect()
    if DEVICE == "xpu":  torch.xpu.empty_cache()
    elif DEVICE == "cuda": torch.cuda.empty_cache()

## STAGE 1 — Data Loading

In [6]:
try:
    df = pd.read_csv(CSV_FILE, encoding='latin1')
    print(f"Loaded {len(df)} rows dari {CSV_FILE}.")
except Exception as e:
    print(f"CSV not found ({e}). Creating dummy data.")
    df = pd.DataFrame({
        'Essay':    ["This is a test essay with good grammar."] * 60,
        'Question': ["Write a test essay."] * 60,
        'graph':    np.random.randint(1, 4, 60),
    })
    for c in TEXT_TRAITS:
        df[c] = np.random.randint(1, 6, size=len(df))

if os.path.exists('../../train_df.csv') and os.path.exists('../../test_df.csv'):
    print("Memuat ../../train_df.csv & ../../test_df.csv (test set konsisten)...")
    train_df = pd.read_csv('../../train_df.csv')
    test_df  = pd.read_csv('../../test_df.csv')
elif os.path.exists('../rq4/train_df.csv') and os.path.exists('../rq4/test_df.csv'):
    print("Memuat train_df.csv & test_df.csv dari ../rq4/ ...")
    train_df = pd.read_csv('../rq4/train_df.csv')
    test_df  = pd.read_csv('../rq4/test_df.csv')
elif os.path.exists('train_df.csv') and os.path.exists('test_df.csv'):
    print("Memuat train_df.csv & test_df.csv lokal...")
    train_df = pd.read_csv('train_df.csv')
    test_df  = pd.read_csv('test_df.csv')
else:
    print("File split belum ada. Melakukan GroupShuffleSplit baru...")
    gss_test = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss_test.split(df, groups=df['graph']))
    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df  = df.iloc[test_idx].reset_index(drop=True)
    train_df.to_csv('train_df.csv', index=False)
    test_df.to_csv('test_df.csv',  index=False)

bert_tok    = AutoTokenizer.from_pretrained("roberta-base")
test_ds     = TextOnlyDataset(test_df, bert_tok)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_df)} rows | Test: {len(test_df)} rows.")

Loaded 1054 rows dari ../../data.csv.
Memuat ../../train_df.csv & ../../test_df.csv (test set konsisten)...
Train: 867 rows | Test: 187 rows.


## STAGE 2 — Loop Training per Pooling Strategy

Untuk setiap pooling: train 100% data train (15 epoch), simpan model + prediksi train/test, hitung QWK per-trait + mean.

In [7]:
criterion_score = nn.MSELoss()
trait_short = [t.replace('(ground_truth)', '') for t in TEXT_TRAITS]
comparison_rows = []

for pooling in POOLING_STRATEGIES:
    print("\n" + "=" * 60)
    print(f"POOLING: {pooling.upper()}")
    print("=" * 60)

    _clear_gpu()
    set_seed(42)
    g_final = torch.Generator(); g_final.manual_seed(42)

    full_train_loader = DataLoader(
        TextOnlyDataset(train_df, bert_tok),
        batch_size=BATCH_SIZE, shuffle=True, generator=g_final,
    )

    final_model     = RobertaSemanticAES(dropout_rate=DROPOUT_RATE, hidden_dim=HIDDEN_DIM, pooling=pooling).to(DEVICE)
    final_optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, final_model.parameters()),
        lr=LEARNING_RATE,
    )

    # ---- TRAIN ----
    for epoch in range(EPOCHS):
        final_model.train()
        loop = tqdm(full_train_loader, desc=f"[{pooling}] Ep {epoch+1}/{EPOCHS}")
        for b in loop:
            final_optimizer.zero_grad()
            ids  = b['input_ids'].to(DEVICE)
            mask = b['attention_mask'].to(DEVICE)
            lbl  = b['labels_txt'].to(DEVICE)

            preds = final_model(ids, mask)
            loss  = criterion_score(preds, lbl)
            loss.backward()
            final_optimizer.step()
            loop.set_postfix({'Loss': f"{loss.item():.4f}"})

    model_path = f"best_roberta_text_pool-{pooling}.pth"
    torch.save(final_model.state_dict(), model_path)
    print(f"  Model -> {model_path}")

    # ---- EXTRACT PREDICTIONS ----
    final_model.eval()

    full_train_loader_noshuffle = DataLoader(
        TextOnlyDataset(train_df, bert_tok), batch_size=BATCH_SIZE, shuffle=False,
    )

    train_preds_arr, test_preds_arr = [], []
    with torch.no_grad():
        for b in tqdm(full_train_loader_noshuffle, desc=f"[{pooling}] Extract Train"):
            ids, mask = b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE)
            train_preds_arr.append(final_model(ids, mask).cpu().numpy())
        for b in tqdm(test_loader, desc=f"[{pooling}] Extract Test"):
            ids, mask = b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE)
            test_preds_arr.append(final_model(ids, mask).cpu().numpy())

    train_preds_arr = np.vstack(train_preds_arr)
    test_preds_arr  = np.vstack(test_preds_arr)

    pd.DataFrame(train_preds_arr, columns=trait_short).to_csv(
        f"roberta_train_text_pool-{pooling}.csv", index=False)
    pd.DataFrame(test_preds_arr,  columns=trait_short).to_csv(
        f"roberta_test_text_pool-{pooling}.csv",  index=False)

    # ---- QWK PER TRAIT (TEST) ----
    per_trait_test = {}
    for ti, (short, gt_col) in enumerate(zip(trait_short, TEXT_TRAITS)):
        pred_vals    = test_preds_arr[:, ti]
        gt_vals      = test_df[gt_col].values
        pred_classes = np.round(pred_vals * 2).astype(int)
        gt_classes   = np.round(gt_vals   * 2).astype(int)
        per_trait_test[short] = cohen_kappa_score(gt_classes, pred_classes, weights='quadratic')

    mean_qwk_test = float(np.mean(list(per_trait_test.values())))

    # mean QWK train (untuk diagnostik overfit)
    per_trait_train = []
    for ti, gt_col in enumerate(TEXT_TRAITS):
        pred_classes = np.round(train_preds_arr[:, ti] * 2).astype(int)
        gt_classes   = np.round(train_df[gt_col].values * 2).astype(int)
        per_trait_train.append(cohen_kappa_score(gt_classes, pred_classes, weights='quadratic'))
    mean_qwk_train = float(np.mean(per_trait_train))

    print(f"  Mean QWK Train : {mean_qwk_train:.4f}")
    print(f"  Mean QWK Test  : {mean_qwk_test:.4f}")
    for short, qwk in per_trait_test.items():
        print(f"    {short:<25s} : {qwk:.4f}")

    row = {'pooling': pooling, 'mean_qwk_train': mean_qwk_train, 'mean_qwk_test': mean_qwk_test}
    for short, qwk in per_trait_test.items():
        row[f'qwk_{short}'] = qwk
    comparison_rows.append(row)
    pd.DataFrame(comparison_rows).to_csv(RESULTS_CSV, index=False)

    del final_model, final_optimizer, full_train_loader, full_train_loader_noshuffle
    _clear_gpu()


POOLING: MAX


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 329.08it/s, Materializing param=encoder.layer.11.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[max] Ep 6/15:  18%|█▊        | 20/109 [00:30<02:16,  1.53s/it, Loss=0.2619]


KeyboardInterrupt: 

## STAGE 3 — Tabel Perbandingan

In [ ]:
df_results = pd.DataFrame(comparison_rows).sort_values('mean_qwk_test', ascending=False).reset_index(drop=True)

print("\n" + "=" * 70)
print("POOLING ABLATION (TEXT-ONLY, 8 TRAITS) — RINGKASAN")
print("=" * 70)

summary_cols = ['pooling', 'mean_qwk_train', 'mean_qwk_test']
try:
    print(df_results[summary_cols].to_markdown(index=False, floatfmt=".4f"))
except Exception:
    print(df_results[summary_cols].to_string(index=False))

print("\nPer-trait QWK (Test):")
trait_cols = [f'qwk_{s}' for s in trait_short]
try:
    print(df_results[['pooling'] + trait_cols].to_markdown(index=False, floatfmt=".4f"))
except Exception:
    print(df_results[['pooling'] + trait_cols].to_string(index=False))

best = df_results.iloc[0]
print("\nBest pooling : {}  (Mean QWK Test = {:.4f})".format(best['pooling'], best['mean_qwk_test']))
print(f"Hasil disimpan ke: {RESULTS_CSV}")